In [1]:
from transformers import AutoModelForCausalLM
import torch
from huggingface_hub import login

login()

In [ ]:
model_id = "google/gemma-3-1b-it"

hf_model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto"
).eval()


In [3]:
print(hf_model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((11

## Try my version

In [4]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

import torch

from src.gemma3_llm import Gemma3ForCausalLM

model_config = {
        'vocab_size': 262144,
        'residual_channel_size': 1152,
        'num_hidden_layers': 26,
        'intermediate_size': 6912,
        'query_dim': 1024,
        'total_kv_dim': 256,
        'head_dimension': 256
    }

my_model = Gemma3ForCausalLM(model_config)
print(my_model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(
      (embedding): Embedding(262144, 1152, padding_idx=0)
    )
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (output_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm()
          (k_norm): Gemma3RMSNorm()
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSN

In [5]:
num_params = sum(p.numel() for p in hf_model.parameters() if p.requires_grad)
print(f"\n--- Total Trainable Parameters: {num_params / 1e9:.3f} Billion ---")

dummy_input = torch.randint(low=0, high=model_config['vocab_size'], size=(2, 10))
print(f"\n--- Running Forward Pass with Input Shape: {dummy_input.shape} ---")

with torch.no_grad():
    logits = my_model(dummy_input)

print(f"--- Output Logits Shape: {logits.shape} ---")
assert logits.shape == (2, 10, model_config['vocab_size'])
print("--- Model assembled and forward pass successful! ---")


--- Total Trainable Parameters: 1.000 Billion ---

--- Running Forward Pass with Input Shape: torch.Size([2, 10]) ---
--- Output Logits Shape: torch.Size([2, 10, 262144]) ---
--- Model assembled and forward pass successful! ---


In [6]:
import collections

hf_state_dict = hf_model.state_dict()
my_state_dict = my_model.state_dict()

print(f"\nOfficial model has {len(hf_state_dict)} tensors.")
print(f"Custom model has {len(my_state_dict)} tensors.")

new_state_dict = collections.OrderedDict()
for hf_key, hf_tensor in hf_state_dict.items():
    my_key = hf_key
    
    if my_key in my_state_dict:
        if my_state_dict[my_key].shape == hf_tensor.shape:
            new_state_dict[my_key] = hf_tensor
        else:
            print(f"Shape mismatch for key: {my_key}. HF: {hf_tensor.shape}, My: {my_state_dict[my_key].shape}")
    else:
        # I changed these layers name
        if my_key.endswith("o_proj.weight"):
            new_state_dict['.'.join(my_key.split('.')[:-2])+".output_proj.weight"] = hf_tensor
        elif my_key == "model.embed_tokens.weight":
            new_state_dict['.'.join(my_key.split('.')[:-1])+".embedding.weight"] = hf_tensor   
        else:
            print(f"Key not found in custom model: {my_key}")

print("\n--- Loading weights into custom model ---")
report = my_model.load_state_dict(new_state_dict, strict=False)

print("\n--- Load Report ---")
if not report.missing_keys and not report.unexpected_keys:
    print("Success! All weights were loaded perfectly.")
else:
    if report.missing_keys:
        print("Missing keys in custom model:", report.missing_keys)
    if report.unexpected_keys:
        print("Unexpected keys from HF model:", report.unexpected_keys)


Official model has 341 tensors.
Custom model has 341 tensors.

--- Loading weights into custom model ---

--- Load Report ---
Success! All weights were loaded perfectly.


In [22]:
print("\n--- Running Verification ---")
# Move models to the same device for comparison
device = "cuda" if torch.cuda.is_available() else "cpu"
hf_model.to(device)
my_model.to(device)

# Create a dummy input
dummy_input = torch.randint(low=0, high=model_config['vocab_size'], size=(1, 15), device=device)

# Get outputs from both models
with torch.no_grad():
    hf_output = hf_model(dummy_input).logits
    my_output = my_model(dummy_input)

# Check if the outputs are very close
# atol (absolute tolerance) might need slight adjustment depending on dtype
are_close = torch.allclose(hf_output, my_output, atol=1e-3)

print(f"Outputs are close: {are_close}")

if are_close:
    print("\nVerification successful! Your model is a correct implementation.")
else:
    print("\nVerification failed. Outputs do not match.")
    # Print max difference to help debug
    diff = torch.max(torch.abs(hf_output - my_output))
    print(f"Max difference between outputs: {diff.item()}")


--- Running Verification ---


IndexError: tuple index out of range

In [21]:
import torch

# --- Setup for Debugging ---
# We only need the 'outputs' dictionary for this reliable method.
outputs = {"my_model": {}, "hf_model": {}}

# Use the original, simple hook that ONLY captures the output.
# This is guaranteed to work.
def get_hook(name, model_type):
    def hook(model, input, output):
        # The 'output' of a layer is what we want.
        # It's not subject to the same positional/keyword argument issues.
        if isinstance(output, tuple):
            outputs[model_type][name] = output[0].detach()
        else:
            outputs[model_type][name] = output.detach()
    return hook

# Use the same dummy input
device = "cuda" if torch.cuda.is_available() else "cpu"
dummy_input = torch.randint(low=0, high=model_config['vocab_size'], size=(1, 15), device=device)

# --- Attach hooks to the OUTPUT of the input_layernorm in the first layer ---
my_norm_hook = my_model.model.layers[0].input_layernorm.register_forward_hook(get_hook("norm_0_output", "my_model"))
hf_norm_hook = hf_model.model.layers[0].input_layernorm.register_forward_hook(get_hook("norm_0_output", "hf_model"))

# --- Run forward pass ---
with torch.no_grad():
    hf_model(dummy_input)
    my_model(dummy_input)

# --- Compare the outputs of the normalization layer ---
# This is the effective input to the attention block
are_attn_inputs_close = torch.allclose(
    outputs["my_model"]["norm_0_output"], 
    outputs["hf_model"]["norm_0_output"], 
    atol=1e-4
)
print(f"Are the inputs to the attention block (i.e., outputs of input_layernorm) close? -> {are_attn_inputs_close}")

if not are_attn_inputs_close:
    diff = torch.max(torch.abs(outputs["my_model"]["norm_0_output"] - outputs["hf_model"]["norm_0_output"]))
    print(f"  - Max difference in attention block INPUT: {diff.item()}")

# --- Remove hooks ---
my_norm_hook.remove()
hf_norm_hook.remove()

IndexError: tuple index out of range